# EE2211 Tutorial 12 — Neural Networks (MLP)

Covers Questions 3, 4, 5.

**Key formula — L-layer network** (with bias prepend at each layer):
$$F_\mathbf{w}(\mathbf{X}) = f\bigl([\mathbf{1},\, f([\mathbf{1},\, f(\mathbf{X}W_1)]W_2)\bigr]W_3)$$
where $[\mathbf{1}, H]$ means prepend a column of ones to $H$ (adds bias term).

**Without bias (Q3)**:
$$F_\mathbf{w}(\mathbf{X}) = f(f(\mathbf{X}W_1)W_2)$$

In [ ]:
import numpy as np

def relu(x):    return np.maximum(0, x)
def sigmoid(x): return 1.0 / (1.0 + np.exp(-x))

## Question 3 — 2-Layer Network, ReLU Activation

$$F_\mathbf{w}(\mathbf{X}) = f(f(\mathbf{X}W_1)W_2)$$

where $\mathbf{X} = \begin{bmatrix}1&1&3.0\\1&2&2.5\end{bmatrix}$, $W_1 = W_2 = \begin{bmatrix}-1&0&1\\0&-1&0\\1&0&1\end{bmatrix}$, activation = ReLU.

In [ ]:
X  = np.array([[1, 1, 3.0], [1, 2, 2.5]])
W1 = np.array([[-1, 0, 1], [0, -1, 0], [1, 0, 1]])
W2 = W1.copy()

H1 = relu(X @ W1)
print('After layer 1 — f(XW1):')
print(np.round(H1, 4))

out = relu(H1 @ W2)
print('\nOutput F_w(X):')
print(np.round(out, 1))
# Expected: [[2, 0, 6], [2, 0, 5]]

## Question 4 — 3-Layer Network, Sigmoid + Bias Column

$$F_\mathbf{w}(\mathbf{X}) = f\bigl([\mathbf{1},\, f([\mathbf{1},\, f(\mathbf{X}W_1)]W_2)\bigr]W_3)$$

$\mathbf{X} = \begin{bmatrix}1&2&1\\1&5&1\end{bmatrix}$,
$W_1 = \begin{bmatrix}-1&0&1\\0&-1&0\\1&0&-1\end{bmatrix}$,
$W_2 = W_3 = \begin{bmatrix}-1&0&1\\0&-1&0\\1&0&1\\1&-1&1\end{bmatrix}$,
activation = Sigmoid.

**Note**: `[1, H]` = `np.hstack([np.ones((n,1)), H])` — prepends bias column.

In [ ]:
X  = np.array([[1, 2, 1], [1, 5, 1]], dtype=float)
W1 = np.array([[-1, 0, 1], [0, -1, 0], [1, 0, -1]], dtype=float)
W2 = np.array([[-1, 0, 1], [0, -1, 0], [1, 0, 1], [1, -1, 1]], dtype=float)
W3 = W2.copy()

n  = X.shape[0]
ones = np.ones((n, 1))

# Layer 1
H1 = sigmoid(X @ W1)
print('f(XW1):', np.round(H1, 4))

# Layer 2 — prepend bias
H1b = np.hstack([ones, H1])
H2  = sigmoid(H1b @ W2)
print('f([1,H1]W2):', np.round(H2, 4))

# Layer 3 — prepend bias
H2b = np.hstack([ones, H2])
out = sigmoid(H2b @ W3)
print('\nOutput F_w(X):')
print(np.round(out, 4))
# Expected: [[0.5259, 0.2243, 0.8913], [0.5219, 0.2319, 0.8897]]

## Question 5 — MLP CV: Find Best Hidden Layer Size on Iris

(a) 80/20 train/test split (`random_state=0`)  
(b) 5-fold CV on training set, sweep `Nhidd` in 1–10 for `(Nhidd, Nhidd, Nhidd)` hidden layers  
(c) Find best `Nhidd` by max validation accuracy  
(d) Evaluate on test set → expected accuracy ≈ 1.0

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn import metrics

iris = load_iris()
X_train, X_test, y_train, y_test = train_test_split(
    iris.data, iris.target, test_size=0.20, random_state=0
)
print(f'Train: {X_train.shape}  Test: {X_test.shape}')

In [ ]:
acc_train_array, acc_valid_array = [], []

# Fixed permutation so folds are reproducible
Idx = np.random.RandomState(seed=8).permutation(len(y_train))  # 120 samples

for Nhidd in range(1, 11):
    fold_tr, fold_va = [], []
    for k in range(5):
        N = int(np.round((k + 1) * len(y_train) / 5))  # end index
        val_idx  = Idx[N - 24 : N]                      # 24 validation samples
        train_idx = np.setdiff1d(Idx, val_idx)

        clf = MLPClassifier(solver='lbfgs', alpha=1e-5,
                            hidden_layer_sizes=(Nhidd, Nhidd, Nhidd),
                            random_state=1, max_iter=500)
        clf.fit(X_train[train_idx], y_train[train_idx])

        fold_tr.append(metrics.accuracy_score(y_train[train_idx], clf.predict(X_train[train_idx])))
        fold_va.append(metrics.accuracy_score(y_train[val_idx],   clf.predict(X_train[val_idx])))

    acc_train_array.append(np.mean(fold_tr))
    acc_valid_array.append(np.mean(fold_va))

best_Nhidd = np.argmax(acc_valid_array) + 1
print(f'Best hidden node size = {best_Nhidd}  (val acc = {acc_valid_array[best_Nhidd-1]:.3f})')

plt.figure(figsize=(8, 5))
plt.plot(range(1, 11), acc_train_array, 'b-o', lw=2, label='Training')
plt.plot(range(1, 11), acc_valid_array, color='orange', marker='x', lw=2, label='Validation')
plt.xlabel('Number of hidden nodes per layer')
plt.ylabel('Accuracy')
plt.title('Training and Validation Accuracies')
plt.xticks(range(1, 11))
plt.legend(); plt.grid(True); plt.tight_layout(); plt.show()

In [ ]:
# (d) Final evaluation on held-out test set
clf_final = MLPClassifier(solver='lbfgs', alpha=1e-5,
                          hidden_layer_sizes=(best_Nhidd, best_Nhidd, best_Nhidd),
                          random_state=1, max_iter=500)
clf_final.fit(X_train, y_train)
test_acc = metrics.accuracy_score(y_test, clf_final.predict(X_test))
print(f'Best hidden node size = {best_Nhidd}  →  Test accuracy = {test_acc:.3f}')
# Expected: test accuracy = 1.0